<a href="https://colab.research.google.com/github/perekis228/DU/blob/main/03tasks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Задание 1. Задачи с цепями

### Модель №1 (соскальзывание без сопротивления)

Однородная цепь длиной 2.45 м соскальзывает с гладкого горизонтального стола. В начальный момент наблюдения со стола свисал конец цепи длиной 0.3 м. Считая ускорение свободного падения равным 9.8 м/с² и пренебрегая трением цепи о поверхность стола, найдите **время** соскальзывания всей цепи со стола, если в начале наблюдения скорость соскальзывания цепи составила 1 м/с.

In [ ]:
from sympy import *

t = symbols('t')
y = Function('y')
g = Rational(98, 10)
L = Rational(245, 100)

eq1 = Eq(L*y(t).diff(t, 2), y(t)*g)
ics = {y(0): Rational(3, 10), y(t).diff(t).subs(t, 0): 1}
y_t = dsolve(eq1, y(t), ics=ics)

eq2 = Eq(L, y_t.rhs)
t_end = [sol.evalf() for sol in solve(eq2) if sol.is_real][0]
t_end

0.909488502410478

### Модель №2 (соскальзывание c сопротивлением)

Однородная цепь длиной 2.45 м соскальзывает с горизонтального стола. Коэффициент трения скольжения цепи о стол μ = 0.1. В начальный момент наблюдения со стола свисает участок цепи длиной 0.3 м, а скорость цепи равна 1 м/с. Сила трения действует на часть цепи, лежащую на столе, и пропорциональна силе её нормального давления на стол (F = μN). Принимая ускорение свободного падения 9.8 м/с², найдите:

1. время t (в секундах), за которое вся цепь соскользнёт со стола, отсчитывая его от начального момента наблюдения;
2. длину (в метрах) свисающего участка, при которой сила тяжести этого участка уравновешивается силой трения, действующей на часть цепи, лежащую на столе.

In [ ]:
t = symbols('t')
y = Function('y')
g = Rational(98, 10)
L = Rational(245, 100)
mu = Rational(1, 10)

eq1 = Eq(L*y(t).diff(t, 2), y(t)*g - mu*(L-y(t))*g)
ics = {y(0): Rational(3, 10), y(t).diff(t).subs(t, 0): 1}
y_t = dsolve(eq1, y(t), ics=ics)

eq2 = Eq(L, y_t.rhs)
t_end = [sol.evalf() for sol in solve(eq2) if sol.is_real][0]

eq3 = Eq(y(t)*g, mu*(L-y(t))*g)
y_balance = solve(eq3, y(t))[0].evalf()
print(t_end, y_balance, sep='\n')

0.998980379872046
0.222727272727273


## Задание 2. Вертикальный взлет метеорологической ракеты

### Модель 1

Ознакомьтесь с [работой](https://doi.org/10.32517/0234-0453-2022-37-5-62-75) по моделированию запуска метеорологической ракеты МР-12.

Используя параметры ракеты, дискретное уменьшение массы топлива и формулу для аэродинамического сопротивления из работы составьте дифференциальное уравнение второго порядка, которое описывает взлет ракеты. Численно решите уравнение для разных масс с использованием `solve_ivp` до момента достижения апогея. Доведите модель до результатов схожих с представленными в работе.

In [1]:
from scipy.integrate import solve_ivp
# параметры ракеты МР-12
M_fuel = 1200.0    # масса топлива (кг)
M_body = 250.0     # масса корпуса (кг)
thrust = 103600.0  # тяга (Н)
burn_time = 24.0   # время работы двигателя (с)
g = 9.8            # ускорение свободного падения (м/с^2)
dt = 0.1           # шаг дискретизации для массы
n = round(burn_time/dt)
dm = M_fuel/n      # расход топлива на шаг

# аэродинамическое сопротивление
def drag(v, h):
    return 0.1 * v * v * 10 ** (-0.0000555 * h)

$$ma=T-mg-F_{сопр}$$
$$T=\begin{cases}
103600 Н,\ 0\leq t<24 с \\
0,\ t\geq 0.
\end{cases}$$
$$k(t)=\min(\lfloor \frac{t}{dt} \rfloor,\ n)$$
$$m(t)=M_{корп}+M{нагр}+M{топл}(1-\frac{k(t)}{n})$$

In [3]:
M_loads = [50, 100, 200]

for M_load in M_loads:
  def T(t):
    if t < 24:
      return 103600
    return 0

  def m(t):
    k = min(int(t/dt), n)
    return M_body + M_load + M_fuel*(1-k/n)

  def apogee_event(t, y):
      return y[1]

  apogee_event.terminal = True #остановит интегрирование, как только событие произойдёт
  apogee_event.direction = -1 #отсекает нулевую скорость в самой начальной точке

  def dynamics(t, y):
      h, v = y
      mt = m(t)
      dhdt = v
      dvdt = (T(t) - mt*g - drag(v, h)) / mt
      return [dhdt, dvdt]

  y0 = [0.0, 0.0]
  t_span = (0, 1000)

  sol = solve_ivp(
      dynamics,
      t_span,
      y0,
      events=apogee_event,
      max_step=dt,
      rtol=1e-9, atol=1e-9 # точность
  )

  h_apogee = sol.y_events[0][0][0]   # высота в момент апогея
  print("Масса:", M_load)
  print("Апогей:", h_apogee / 1000, "км")
  print("Время апогея:", sol.t_events[0][0], "с\n")

Масса: 50
Апогей: 175.99707531502713 км
Время апогея: 201.448709124069 с

Масса: 100
Апогей: 157.99158513731237 км
Время апогея: 191.34786782518913 с

Масса: 200
Апогей: 130.27562761761544 км
Время апогея: 174.59437855941857 с



### Модель 2

Представленная в работе модель аэродинамического сопротивления очень упрощённая и является просто эмпирическим приближением. В реальности сопротивление зависит от плотности атмосферы, площади поперечного сечения ракеты и коэффициента аэродинамического сопротивления:

$${\displaystyle F_{\mathrm {D} }\,=\,{\tfrac {1}{2}}\,\rho \,v^{2}\,C_{\mathrm {D} }\,A}$$
где
* $\rho(h)$ - плотность воздуха на высоте $h$,
* $C_{D}$ - коэффициент аэродинамического сопротивления,
* $A$ - площадь поперечного сечения,
* $v$ - скорость ракеты.

Известен ["калибр"](https://ru.wikipedia.org/wiki/%D0%9C%D0%A0-12) ракеты, который равен 450 мм.

В атмосфере Земли плотность атмосферы падает [почти экспоненциально](https://en.wikipedia.org/wiki/International_Standard_Atmosphere) с высотой.

Составьте физически более правильную модель изменения плотности воздуха и, соответственно, аэродинамического сопротивления, при которой сопротивление будет уменьшаться экспоненциально с высотой и на больших высотах станет практически нулевым, как это происходит в реальности.

Поскольку параметр $A$ может быть вычислен, выполните оценку коэффициента аэродинамического сопротивления ракеты для достижения аналогичных результатов.

$$A=\frac{\pi D^2}{4} = \frac{3.14159 \cdot 0.45^2}{4} \approx 0.159\ m^2$$
$$\rho(h)=\rho_0 \cdot e^{-\frac{h}{H}},\ \rho_0=1.1245\ kg/m^3,\ H=7825\ m$$
$$F_D=\frac{1}{2}\rho_0 e^{-\frac{h}{H}} v^2 C_D A,\ C_D\approx1$$

In [4]:
import numpy as np
from scipy.integrate import solve_ivp

# параметры ракеты МР-12
M_fuel = 1200.0    # масса топлива (кг)
M_body = 250.0     # масса корпуса (кг)
thrust = 103600.0  # тяга (Н)
burn_time = 24.0   # время работы двигателя (с)
g = 9.8            # ускорение свободного падения (м/с^2)
dt = 0.1           # шаг дискретизации для массы
n = round(burn_time/dt)
dm = M_fuel/n      # расход топлива на шаг

rho_0 = 1.225          # плотность воздуха на уровне моря (кг/м^3)
H = 7825.0             # высота однородной атмосферы (м)
D = 0.45               # калибр ракеты (м)
A = np.pi * D**2 / 4   # площадь поперечного сечения (м^2)
C_D = 1.0             # коэффициент аэродинамического сопротивления

# аэродинамическое сопротивление
def drag(v, h):
    rho = rho_0 * np.exp(-h / H)
    return 0.5 * rho * v * v * C_D * A

M_loads = [50, 100, 200]

for M_load in M_loads:
  def T(t):
    if t < 24:
      return 103600
    return 0

  def m(t):
    k = min(int(t/dt), n)
    return M_body + M_load + M_fuel*(1-k/n)

  def apogee_event(t, y):
      return y[1]

  apogee_event.terminal = True #остановит интегрирование, как только событие произойдёт
  apogee_event.direction = -1 #отсекает нулевую скорость в самой начальной точке

  def dynamics(t, y):
      h, v = y
      mt = m(t)
      dhdt = v
      dvdt = (T(t) - mt*g - drag(v, h)) / mt
      return [dhdt, dvdt]

  y0 = [0.0, 0.0]
  t_span = (0, 1000)

  sol = solve_ivp(
      dynamics,
      t_span,
      y0,
      events=apogee_event,
      max_step=dt,
      rtol=1e-9, atol=1e-9 # точность
  )

  h_apogee = sol.y_events[0][0][0]   # высота в момент апогея
  print("Масса:", M_load)
  print("Апогей:", h_apogee / 1000, "км")
  print("Время апогея:", sol.t_events[0][0], "с\n")

Масса: 50
Апогей: 181.37966317824632 км
Время апогея: 204.47701671614092 с

Масса: 100
Апогей: 162.46815046296564 км
Время апогея: 194.01736481781245 с

Масса: 200
Апогей: 133.51965440596697 км
Время апогея: 176.7431798971938 с

